# Notebook 04 — Uplift by Customer Segment

## What is Uplift?

**Uplift** (also called *incremental lift*) measures the additional conversion rate caused by the promotion, *above what would have happened organically*.

$$\text{Uplift}_{\text{absolute}} = \text{Conversion}_{\text{treatment}} - \text{Conversion}_{\text{control}}$$
$$\text{Uplift}_{\text{relative}} = \frac{\text{Uplift}_{\text{absolute}}}{\text{Conversion}_{\text{control}}} \times 100\%$$

**Why segment?** The same promotion can have very different effects on different customer groups. A 10% average uplift might hide a 25% uplift in one segment and 0% in another — knowing this allows budget reallocation toward the highest-responders.

**Segments analyzed:** gender, age group, income band, and combinations.

In [1]:
import sys
from pathlib import Path
_root = Path().resolve(); _root = _root.parent if _root.name == 'notebooks' else _root; sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.extract import extract
from src.transform import transform
from src.constants import REPORTS_FIGURES, ALPHA
from src.utils.stats import ab_test_proportions
from src.utils.plot import save_fig, uplift_heatmap

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

portfolio, profile, transcript = extract()
tables = transform(portfolio, profile, transcript)
master = tables['master_table']

## 1. Uplift Calculation Function

In [2]:
def compute_uplift_by_segment(
    df: pd.DataFrame,
    segment_col: str,
    offer_type: str | None = None,
    min_n: int = 50,
) -> pd.DataFrame:
    """Compute uplift metrics for each value of segment_col.

    Segments with fewer than min_n observations in either group are excluded
    to avoid unreliable estimates.
    """
    if offer_type:
        df = df[df['offer_type'] == offer_type]

    rows = []
    for seg_val, grp in df.groupby(segment_col, observed=True):
        ctrl = grp[grp['viewed'] == False]['completed']
        trt  = grp[grp['viewed'] == True]['completed']
        if len(ctrl) < min_n or len(trt) < min_n:
            continue
        res = ab_test_proportions(ctrl, trt, alpha=ALPHA)
        rows.append({
            'segment': str(seg_val),
            'n_control': res['n_control'],
            'n_treatment': res['n_treatment'],
            'conv_control': res['conversion_control'],
            'conv_treatment': res['conversion_treatment'],
            'uplift_abs': res['uplift_absolute'],
            'uplift_rel_pct': res['uplift_relative_pct'],
            'p_value': res['p_value'],
            'significant': res['significant'],
        })
    return pd.DataFrame(rows).sort_values('uplift_rel_pct', ascending=False)

## 2. Uplift by Gender

In [3]:
gender_uplift = compute_uplift_by_segment(master, 'gender')
gender_uplift

,segment,n_control,n_treatment,conv_control,conv_treatment,uplift_abs,uplift_rel_pct,p_value,significant
2,O,130,1370,0.3462,0.7073,0.3611,104.33,0.0,True
1,M,8850,48401,0.3129,0.6259,0.3130,100.04,0.0,True
0,F,6101,37841,0.4868,0.7385,0.2517,51.71,0.0,True


In [4]:
colors = ['#00704A' if sig else '#C0C0C0' for sig in gender_uplift['significant']]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(gender_uplift['segment'], gender_uplift['uplift_rel_pct'], color=colors)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Uplift by Gender (green = statistically significant at α=0.05)')
ax.set_ylabel('Relative Uplift (%)')
for bar, val in zip(bars, gender_uplift['uplift_rel_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%', ha='center', fontsize=9)
fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '04_uplift_gender.png')
plt.show()

## 3. Uplift by Age Group

In [5]:
age_uplift = compute_uplift_by_segment(master, 'age_group')
age_uplift

,segment,n_control,n_treatment,conv_control,conv_treatment,uplift_abs,uplift_rel_pct,p_value,significant
2,35-44,1712,11981,0.3020,0.6566,0.3546,117.43,0.0,True
1,25-34,1752,7185,0.2945,0.5985,0.3039,103.20,0.0,True
0,<25,1283,5244,0.2899,0.5730,0.2831,97.64,0.0,True
3,45-54,2966,18898,0.3955,0.6880,0.2925,73.95,0.0,True
4,55-64,3348,20236,0.4337,0.7014,0.2677,61.72,0.0,True
5,65+,4020,24068,0.4363,0.6998,0.2634,60.38,0.0,True


## 4. Uplift by Income Band

In [6]:
income_uplift = compute_uplift_by_segment(master, 'income_group')
income_uplift

,segment,n_control,n_treatment,conv_control,conv_treatment,uplift_abs,uplift_rel_pct,p_value,significant
0,low,2782,10807,0.2552,0.5517,0.2965,116.17,0.0,True
1,mid,4758,25971,0.2959,0.6166,0.3207,108.37,0.0,True
2,high,4132,28393,0.4134,0.6982,0.2849,68.92,0.0,True
3,premium,3409,22441,0.5744,0.7757,0.2013,35.05,0.0,True


## 5. Heatmap — Offer Type × Age Group

This view shows which (offer type, age group) combinations drive the most uplift. Cells with the highest positive values in green are the best targeting opportunities.

In [7]:
rows = []
for otype in ['bogo', 'discount']:
    df_type = master[master['offer_type'] == otype]
    for age_val, grp in df_type.groupby('age_group', observed=True):
        ctrl = grp[grp['viewed'] == False]['completed']
        trt  = grp[grp['viewed'] == True]['completed']
        if len(ctrl) < 30 or len(trt) < 30:
            continue
        res = ab_test_proportions(ctrl, trt)
        rows.append({'offer_type': otype, 'age_group': str(age_val), 'uplift_pct': res['uplift_relative_pct']})

pivot = pd.DataFrame(rows).pivot(index='offer_type', columns='age_group', values='uplift_pct')
fig = uplift_heatmap(
    pivot,
    title='Relative Uplift (%) — Offer Type × Age Group',
    save_path=REPORTS_FIGURES / '04_uplift_heatmap.png',
)
plt.show()

## 6. Top 10 Significant Segments — Full Ranking

In [8]:
all_rows = []
for otype in ['bogo', 'discount']:
    for seg_col in ['gender', 'age_group', 'income_group']:
        df_seg = compute_uplift_by_segment(master, seg_col, offer_type=otype)
        df_seg['offer_type'] = otype
        df_seg['dimension'] = seg_col
        all_rows.append(df_seg)

ranking = (
    pd.concat(all_rows)
    .query('significant == True')
    .sort_values('uplift_rel_pct', ascending=False)
    .head(10)
    [['offer_type', 'dimension', 'segment', 'conv_control', 'conv_treatment', 'uplift_rel_pct', 'p_value']]
)
print('Top 10 Segments by Uplift (statistically significant only):')
ranking

Top 10 Segments by Uplift (statistically significant only):


,offer_type,dimension,segment,conv_control,conv_treatment,uplift_rel_pct,p_value
0,discount,income_group,low,0.2896,0.7801,169.37,0.0
2,discount,age_group,35-44,0.3471,0.8545,146.17,0.0
1,discount,age_group,25-34,0.3348,0.8163,143.84,0.0
0,discount,age_group,<25,0.3407,0.7920,132.44,0.0
1,discount,income_group,mid,0.3601,0.8154,126.45,0.0
1,discount,gender,M,0.3736,0.8208,119.68,0.0
2,discount,gender,O,0.4697,0.8693,85.09,0.0
2,discount,income_group,high,0.5002,0.8718,74.28,0.0
3,discount,age_group,45-54,0.5036,0.8548,69.74,0.0
4,discount,age_group,55-64,0.5258,0.8634,64.22,0.0


## 7. Key Findings

*(Fill in after running the notebook)*

- **Best segment overall:** [offer_type] in [segment] → +X% uplift (p = Y)
- **BOGO peak segment:** [age_group] [gender] → +X%
- **Discount peak segment:** [income_group] → +X%
- **No significant uplift:** [segments where promotion had no effect]

---
**Next:** Notebook 05 converts these uplift numbers into business ROI.